In [0]:
# Analytical lakemarts
lakeName2 = "udalstdataanalysisprod.dfs.core.windows.net" # lakemart path
containerName4 = "analytics-projects" # high level folder

# Combine path
path = "abfss://"+containerName4+"@"+lakeName2+"/StrategyUnit/StrategyUnit/analytics_modelling/bbcs_maternity/final_cohort_2326.parquet/"

# Read in from parquet file
final_cohort_2326= spark.read.option("header", "true").option("recursiveFileLookup", "True").parquet(path)

# Check py_spark table
display(final_cohort_2326.limit(10))

# Write as temporary sql view
final_cohort_2326.createOrReplaceTempView("final_cohort_2326")


In [0]:

trust_to_icb_lookup= spark.read.table("udal_gold_restricted.ukhd_ods.Provider_Hierarchies_ICB")

trust_to_icb_lookup.createOrReplaceTempView("trust_to_icb_lookup")


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW trust_matrix AS 

SELECT 
 CASE WHEN birth_provider_flag = 1 THEN birth_provider ELSE 'Other' END AS birth_provider_grouped,
  CASE WHEN booking_provider_flag = 1 THEN booking_provider ELSE 'Other' END AS booking_provider_grouped,
  COUNT(*) AS n

FROM final_cohort_2326

WHERE birth_flag=1 AND
birth_yr LIKE '2025/26'

GROUP BY 
  CASE WHEN birth_provider_flag = 1 THEN birth_provider ELSE 'Other' END,
  CASE WHEN booking_provider_flag = 1 THEN booking_provider ELSE 'Other' END

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW final_cohort_with_icb_data AS 

SELECT 
  CASE 
    WHEN a.resident_icb IS NOT NULL THEN a.resident_icb
    WHEN a.resident_icb IS NULL AND a.la_of_mother IN ('E08000025', 'E08000029' ) THEN 'QHL'
    WHEN a.resident_icb IS NULL AND a.la_of_mother IN ('E08000030', 'E08000031', 'E08000027','E08000028') THEN 'QUA'
    ELSE NULL  ---- try to fill missing resident_icb using la if available
  END AS resident_icb,
  a.birth_provider,
  b.ICB_Code AS birth_icb,
  b.Integrated_Care_Board_Name AS birth_icb_name

FROM final_cohort_2326 a

LEFT JOIN trust_to_icb_lookup b
    ON a.birth_provider = b.Organisation_Code

WHERE a.birth_flag = 1
  AND birth_yr LIKE '2025/26'

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW icb_matrix AS 
SELECT 
    CASE WHEN resident_icb IN ('QHL', 'QUA', 'QWU', 'QJ2', 'QGH', 'QOC', 'QNC') THEN resident_icb
      ELSE 'Other'  END AS resident_icb_interest,
  CASE WHEN  birth_icb IN ('QHL', 'QUA', 'QWU', 'QJ2', 'QGH', 'QOC', 'QNC') THEN birth_icb
      ELSE 'Other'  END AS birth_icb_interest,
  COUNT(*) AS n

FROM final_cohort_with_icb_data 

GROUP BY 
    CASE WHEN resident_icb IN ('QHL', 'QUA', 'QWU', 'QJ2', 'QGH', 'QOC', 'QNC') THEN resident_icb
      ELSE 'Other'  END,
  CASE WHEN  birth_icb IN ('QHL', 'QUA', 'QWU', 'QJ2', 'QGH', 'QOC', 'QNC') THEN birth_icb
      ELSE 'Other'  END 

In [0]:
#Read table into a dataframe for use in python
trust_matrix = spark.table("trust_matrix")
icb_matrix = spark.table("icb_matrix")

# set path to lake mart
lake = "udalstdataanalysisprod.dfs.core.windows.net"
container = "analytics-projects"
folder = "/StrategyUnit/StrategyUnit/analytics_modelling/bbcs_maternity/"
eval_path = "abfss://" + container + "@" + lake + folder

# save to SU lake mart
trust_matrix.write.mode("overwrite").parquet(eval_path + "trust_matrix.parquet")
icb_matrix.write.mode("overwrite").parquet(eval_path + "icb_matrix.parquet")
